In [ ]:
# Install necessary libraries
!pip install tensorflow matplotlib kaggle

import os
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, LearningRateScheduler
import matplotlib.pyplot as plt

# Function to prepare dataset directories
def prepare_dataset(dataset_path):
    if not os.path.exists(dataset_path):
        print(f"Error: Could not find the dataset file.")
        print(f"Expected path: {dataset_path}")
        print("Please ensure the dataset is correctly placed.")
        return None, None
    
    with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
        zip_ref.extractall('dermnet')
    
    base_dir = 'dermnet'
    train_dir = os.path.join(base_dir, 'train')
    test_dir = os.path.join(base_dir, 'test')

    if not os.path.exists(train_dir) or not os.path.exists(test_dir):
        print(f"Error: Could not find the train or test directories.")
        print(f"Expected paths: {train_dir}, {test_dir}")
        print("Please ensure the dataset is correctly structured.")
        return None, None
    
    return train_dir, test_dir

# Specify the path to your dataset
dataset_path = 'data/dermnet.zip'  # Replace this with your actual dataset path

# Prepare the dataset
train_dir, test_dir = prepare_dataset(dataset_path)

if train_dir is None or test_dir is None:
    print("Failed to prepare the dataset. Please resolve the issues and run the script again.")
    import sys
    sys.exit(1)

# Data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Build the model using EfficientNetB3 as base
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
base_model.trainable = False

# Add custom layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

# Compile the model with an initial learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.losses.binary_crossentropy,
    metrics=[tf.keras.metrics.BinaryAccuracy(name = "accuracy")]
)

model.summary()

# Callbacks
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True, mode='min')
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

# Learning Rate Scheduler
def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

lr_scheduler = LearningRateScheduler(scheduler)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    callbacks=[checkpoint, reduce_lr, early_stopping, lr_scheduler]
)

# Fine-tuning
base_model.trainable = True

# Recompile the model with a lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss=tf.losses.binary_crossentropy,
    metrics=[tf.keras.metrics.BinaryAccuracy(name = "accuracy")]
)

# Continue training
history_fine = model.fit(
    train_generator,
    epochs=50,
    validation_data=test_generator,
    callbacks=[checkpoint, reduce_lr, early_stopping, lr_scheduler]
)

# Load the best model
model.load_weights('best_model.keras')

# Evaluate the model on the test set
val_loss, val_acc = model.evaluate(test_generator)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_acc}")

# Plot training & validation accuracy values
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy (Pre-Fine-tuning)')
plt.plot(history_fine.history['accuracy'], label='Train Accuracy (Fine-tuning)')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy (Pre-Fine-tuning)')
plt.plot(history_fine.history['val_accuracy'], label='Validation Accuracy (Fine-tuning)')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss (Pre-Fine-tuning)')
plt.plot(history_fine.history['loss'], label='Train Loss (Fine-tuning)')
plt.plot(history.history['val_loss'], label='Validation Loss (Pre-Fine-tuning)')
plt.plot(history_fine.history['val_loss'], label='Validation Loss (Fine-tuning)')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TensorFlow Lite model
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)